In [16]:
import grc_utils
import importlib
importlib.reload(grc_utils)
grc_utils.inject_framework_style()

# Module 3: Why AI Risk Is Not Software Risk: Probabilistic Failures

*Enterprise AI Governance, Risk & Control Expert Workbook*

---

In [17]:
grc_utils.display_deliverable_anchor('RSK-03', 'AI vs. Software Risk Divergence Analysis')

## 1. The Fundamental Paradigm Shift

Traditional software governance is built on the foundation of **Determinism**. In software, logic is encoded explicitly; given input $X$, the program will execute instruction set $Y$ to produce output $Z$. Failures are generally binary (it works or it crashes) and logic-driven (a "bug").

AI Governance introduces **Probabilistic Risk**. The logic of an AI system is not explicitly coded but *emerges* from patterns in training data. This creates four critical divergence points defined in the EAGCF Framework:

1.  **Observability Gap**: Software logic (source code) is human-readable and auditable. AI logic (weights/biases) is a high-dimensional "black box."
2.  **Boundary Fragility**: Software has rigid logic gates (if/else). AI has a "Jagged Frontier" where it may perform expertly on task A but fail catastrophically on a slightly perturbed task A'.
3.  **Data-Code Entanglement**: In software, code and data are separate. In AI, the data *is* the logic. Data poisoning or drift directly mutates the system's reasoning.
4.  **Silent Failures**: Software typically fails loudly (exceptions, hangups). AI fails silently (hallucination, bias, drift) while maintaining high confidence in its output.

### Deep Comparative Taxonomy: AI vs. Traditional Software

| Dimension | Traditional Software Risk | AI / Generative AI Risk |
| :--- | :--- | :--- |
| **Core Logic** | Deterministic (Rules-based) | Probabilistic (Pattern-based) |
| **Failure Mode** | Discrete "Bugs" (Logical errors) | Distributional "Drift" & Hallucination |
| **Unit of Audit** | Source Code & Logic Paths | Data Lineage, Weights & Prompts |
| **Security** | Exploits (Buffer overflow, Injection) | Adversarial Attacks (Prompt Injection, Extraction) |
| **Maintenance** | Version Upgrades / Patching | Constant Monitoring / Reinforcement Learning |
| **Explainability** | High (Step-by-step Traceability) | Low (Latent Space Complexity) |
| **Boundary** | Fixed (Hard Input Validation) | Adaptive (Soft, High-Dimensional Frontiers) |

## 2. Practical Simulation: Deterministic vs. Probabilistic Checking

In this section, we simulate a simple Loan Approval system to demonstrate how a software-based check fails predictably versus how an AI-based system can fail unpredictably due to "data drift" or "adversarial perturbations."

In [18]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

def deterministic_validator(income, credit_score):
    """Traditional Software Logic: Hard Rules."""
    if income > 50000 and credit_score > 650:
        return "Approved", 1.0
    return "Rejected", 1.0

def probabilistic_model(income, credit_score, noise=0.1):
    """Simulated AI Model: Pattern Detection with inherent uncertainty."""
    # Base probability calculation
    prob = (income / 100000) * 0.6 + (credit_score / 850) * 0.4
    # Add stochastic noise (simulating 'Jagged Frontier')
    prob += np.random.normal(0, noise)
    prob = np.clip(prob, 0, 1)
    
    decision = "Approved" if prob > 0.65 else "Rejected"
    return decision, prob

# Scenario: Marginal Applicant (Income 51k, Credit 651)
income, credit = 51000, 651

soft_dec, soft_conf = deterministic_validator(income, credit)
ai_dec, ai_conf = probabilistic_model(income, credit)

display(Markdown(f"""
### Scenario Analysis: Marginal Applicant
*   **Software Decision**: {soft_dec} (Confidence: {soft_conf*100}%)
*   **AI Decision**: {ai_dec} (Confidence: {ai_conf*100:.2f}%)
"""))


### Scenario Analysis: Marginal Applicant
*   **Software Decision**: Approved (Confidence: 100.0%)
*   **AI Decision**: Rejected (Confidence: 59.41%)


## 3. The "Jagged Frontier" of AI Risk

The simulation below demonstrates why AI validation requires **Distributional Testing** rather than unit testing. We visualize the "Decision Surface" of the deterministic vs. probabilistic system.

> [!NOTE]
> Notice how the Software logic is a flat plane with a sharp cliff, while the AI logic has ripples and "blind spots" where it might fluctuate between approval and rejection for nearly identical inputs.

In [19]:
# Generate surface data
x = np.linspace(30000, 80000, 50)
y = np.linspace(500, 800, 50)
X, Y = np.meshgrid(x, y)

# Calculate decisions
Z_soft = np.array([[1 if deterministic_validator(ix, iy)[0] == 'Approved' else 0 for ix in x] for iy in y])
Z_ai = np.array([[probabilistic_model(ix, iy, noise=0.02)[1] for ix in x] for iy in y])

fig = make_subplots(
    rows=1, cols=2, 
    specs=[[{'type': 'surface'}, {'type': 'surface'}]],
    subplot_titles=("Software Decision Surface (Deterministic)", "AI Confidence Surface (Probabilistic)")
)

fig.add_trace(go.Surface(z=Z_soft, x=X, y=Y, colorscale='Viridis', showscale=False), row=1, col=1)
fig.add_trace(go.Surface(z=Z_ai, x=X, y=Y, colorscale='Plasma', showscale=False), row=1, col=2)

fig.update_layout(
    title="Risk Topology: Deterministic Rules vs. Probabilistic Latent Space",
    height=600, width=1100,
    scene1=dict(xaxis_title="Income", yaxis_title="Credit", zaxis_title="Approved?"),
    scene2=dict(xaxis_title="Income", yaxis_title="Credit", zaxis_title="Confidence"),
    font=dict(family="Inter")
)
fig.show()

## 4. Shadow AI vs. Shadow IT: The Governance Challenge

Shadow IT (unauthorized use of SaaS/Software) is a major risk, but **Shadow AI** presents a deeper threat:

*   **Data Exfiltration**: Employees pasting PII into public LLMs is easier and harder to detect than installing unauthorized software.
*   **Hallucination in Production**: A user might use ChatGPT to generate code which is then committed to production. The software "works" but contains hidden logic flaws introduced by the LLM.
*   **Intellectual Property Contamination**: Using AI models trained on unlicensed data to generate commercial assets creates a "Legal Debt" that traditional software rarely encounters.

In [20]:
# Summary Visualization: Risk Magnification Metrics
risks = ['Logic Flaws', 'Data Leakage', 'Regulatory Non-compliance', 'Explainability Gap', 'Adversarial Fragility']
software_risk = [60, 40, 50, 20, 10]
ai_risk = [85, 95, 90, 80, 95]

fig = go.Figure()
fig.add_trace(go.Scatterpolar(r=software_risk, theta=risks, fill='toself', name='Traditional Software'))
fig.add_trace(go.Scatterpolar(r=ai_risk, theta=risks, fill='toself', name='Artificial Intelligence'))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title="Risk Magnitude Profile: AI vs. Software",
    font=dict(family="Inter")
)
fig.show()